# **BAGGING**
### **NOMBRE:** Carlos Enrique Nieves Ochoa
### **EXPEDIENTE:** 758210
### **FECHA:** 23 de Noviembre de 2026

### **Librerías**

In [1]:
from sklearn.utils import resample
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

### **Quitamos la columna model del dataset**

In [2]:
df = pd.read_excel("Motor Trend Car Road Tests (1).xlsx")
df = df.drop(columns=['model'])

Me quedo con 32 autos y 10 características para explicar `mpg`. Quito `model` porque solo es el nombre de cada auto y no aporta información numérica al modelo.

### **Crear 10 árboles y=mpg con 3 características al azar (sin repetición) con bootstrap**

In [3]:
X = df.drop(columns="mpg")
y = df["mpg"]

trees = []
np.random.seed(42)
for _ in range(10):
    features = np.random.choice(X.columns, 3, replace=False)
    X_resampled, y_resampled = resample(X[features], y)
    tree = DecisionTreeRegressor(random_state=42)
    tree.fit(X_resampled, y_resampled)
    trees.append((tree, features,))


In [4]:
trees

[(DecisionTreeRegressor(random_state=42),
  array(['gear', 'disp', 'qsec'], dtype=object)),
 (DecisionTreeRegressor(random_state=42),
  array(['disp', 'qsec', 'wt'], dtype=object)),
 (DecisionTreeRegressor(random_state=42),
  array(['cyl', 'vs', 'am'], dtype=object)),
 (DecisionTreeRegressor(random_state=42),
  array(['disp', 'qsec', 'hp'], dtype=object)),
 (DecisionTreeRegressor(random_state=42),
  array(['am', 'drat', 'disp'], dtype=object)),
 (DecisionTreeRegressor(random_state=42),
  array(['am', 'carb', 'cyl'], dtype=object)),
 (DecisionTreeRegressor(random_state=42),
  array(['carb', 'qsec', 'gear'], dtype=object)),
 (DecisionTreeRegressor(random_state=42),
  array(['am', 'disp', 'wt'], dtype=object)),
 (DecisionTreeRegressor(random_state=42),
  array(['qsec', 'drat', 'wt'], dtype=object)),
 (DecisionTreeRegressor(random_state=42),
  array(['gear', 'wt', 'disp'], dtype=object))]

Cada árbol se entrena con una muestra bootstrap distinta y con su propia combinación de tres características. `disp` aparece en 6 de los 10 árboles y `qsec` en 5, mientras que `hp` y `vs` solo entran una vez, así que el promedio depende más de unas variables que de otras por puro azar en el sorteo.

### **Promedio de los 10 árboles y R cuadrada**

In [5]:
y_bag = 0
for tree, features in trees:
    y_bag = y_bag + tree.predict(X[features])
y_bag = y_bag / 10

residuos_bag = y - y_bag
r_cuadrada_bag = 1 - (np.sum(residuos_bag**2) / np.sum((y - np.mean(y))**2))
print("R cuadrada promedio de árboles:", r_cuadrada_bag)

R cuadrada promedio de árboles: 0.9578591660863505


Con el promedio de los 10 árboles obtengo una R cuadrada de 0.958, es decir, explico cerca del 96% de la variación de `mpg`. Aunque el valor es alto, lo calculo con los mismos 32 autos con los que se entrenaron los árboles, y cada árbol crece sin límite de profundidad, por lo que esta cifra es optimista.

### **Regresión lineal múltiple**

In [6]:
modelo = LinearRegression()
modelo.fit(X, y)

r_cuadrada = modelo.score(X, y)
print("R cuadrada regresión lineal:", r_cuadrada)

R cuadrada regresión lineal: 0.8690157644777647


La regresión lineal múltiple con las 10 características alcanza una R cuadrada de 0.869, así que explica cerca del 87% de la variación del rendimiento. También la mido sobre los datos de entrenamiento, aunque al ser un modelo lineal tiene menos capacidad para memorizar cada auto.

### **Comparación**

In [7]:
pd.DataFrame({
    "Modelo": ["Promedio de 10 árboles", "Regresión lineal múltiple"],
    "R cuadrada": [r_cuadrada_bag, r_cuadrada]
})

,Modelo,R cuadrada
0,Promedio de 10 árboles,0.957859
1,Regresión lineal múltiple,0.869016


El promedio de los 10 árboles supera a la regresión lineal por casi 0.09 de R cuadrada (0.958 frente a 0.869), ya que los árboles captan relaciones no lineales entre el rendimiento y variables como `disp` o `wt`. Como ambos modelos se evalúan con los mismos datos de entrenamiento y solo hay 32 autos, la ventaja del bagging podría reducirse si los comparo con datos que no vieron.

### **Train-test 70/30**

Separo el 70% de los autos para entrenar y el 30% para probar, así comparo los dos modelos con datos que no vieron.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

trees_train = []
np.random.seed(42)
for _ in range(10):
    features = np.random.choice(X.columns, 3, replace=False)
    X_resampled, y_resampled = resample(X_train[features], y_train)
    tree = DecisionTreeRegressor(random_state=42)
    tree.fit(X_resampled, y_resampled)
    trees_train.append((tree, features,))

y_bag_test = 0
for tree, features in trees_train:
    y_bag_test = y_bag_test + tree.predict(X_test[features])
y_bag_test = y_bag_test / 10

residuos_bag_test = y_test - y_bag_test
r_cuadrada_bag_test = 1 - (np.sum(residuos_bag_test**2) / np.sum((y_test - np.mean(y_test))**2))
print("R cuadrada promedio de árboles (test):", r_cuadrada_bag_test)

R cuadrada promedio de árboles (test): 0.8623633926811255


In [9]:
modelo_train = LinearRegression()
modelo_train.fit(X_train, y_train)
r_cuadrada_test = modelo_train.score(X_test, y_test)
print("R cuadrada regresión lineal (test):", r_cuadrada_test)

pd.DataFrame({
    "Modelo": ["Promedio de 10 árboles", "Regresión lineal múltiple"],
    "R cuadrada (test)": [r_cuadrada_bag_test, r_cuadrada_test]
})

R cuadrada regresión lineal (test): 0.8040132391853168


,Modelo,R cuadrada (test)
0,Promedio de 10 árboles,0.862363
1,Regresión lineal múltiple,0.804013


En test el promedio de los árboles baja a 0.862 y la regresión a 0.804, así que la ventaja de los árboles se reduce de 0.09 a 0.058. Los árboles sin límite de profundidad memorizaban más los datos de entrenamiento, aunque siguen prediciendo mejor que la regresión.

### **Random forest con train-test 70/30**

Comparo con un `RandomForestRegressor` de 10 árboles con la misma partición. A diferencia de mi ciclo, que fija 3 variables para todo el árbol, el random forest sortea 3 variables nuevas en cada división.

In [10]:
bosque = RandomForestRegressor(n_estimators=10, max_features=3, bootstrap=True, random_state=42)
bosque.fit(X_train, y_train)
r_cuadrada_bosque = bosque.score(X_test, y_test)
print("R cuadrada random forest (test):", r_cuadrada_bosque)

pd.DataFrame({
    "Modelo": ["Promedio de 10 árboles", "Random forest", "Regresión lineal múltiple"],
    "R cuadrada (test)": [r_cuadrada_bag_test, r_cuadrada_bosque, r_cuadrada_test]
})

R cuadrada random forest (test): 0.8798761846754999


,Modelo,R cuadrada (test)
0,Promedio de 10 árboles,0.862363
1,Random forest,0.879876
2,Regresión lineal múltiple,0.804013


El random forest queda arriba con 0.880, seguido del promedio de mis árboles con 0.862 y de la regresión con 0.804. Tiene sentido que gane, ya que sortear variables en cada división hace que los árboles se parezcan menos entre sí y el promedio se equivoque menos.